# EDA — Tennis Match Charting Project
Datasets: `matches`, `Overview`, `Rally`, `KeyPointsServe`, `KeyPointsReturn`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

BASE = 'new-dataset/tennis_MatchChartingProject-master'

matches   = pd.read_csv(f'{BASE}/charting-m-matches.csv')
overview  = pd.read_csv(f'{BASE}/charting-m-stats-Overview.csv')
rally     = pd.read_csv(f'{BASE}/charting-m-stats-Rally.csv')
kp_serve  = pd.read_csv(f'{BASE}/charting-m-stats-KeyPointsServe.csv')
kp_return = pd.read_csv(f'{BASE}/charting-m-stats-KeyPointsReturn.csv')

print('matches:   ', matches.shape)
print('overview:  ', overview.shape)
print('rally:     ', rally.shape)
print('kp_serve:  ', kp_serve.shape)
print('kp_return: ', kp_return.shape)

## 1. Matches

In [ ]:
matches.head(3)

In [ ]:
print(matches.dtypes)
print('\nNulos:')
print(matches.isnull().sum())

In [ ]:
matches['Date'] = pd.to_datetime(matches['Date'], format='%Y%m%d', errors='coerce')
matches['Year'] = matches['Date'].dt.year

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

matches['Surface'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Partidas por Superfície')
axes[0].set_xlabel('')

matches['Round'].value_counts().head(10).plot(kind='bar', ax=axes[1], color='salmon')
axes[1].set_title('Top 10 Rounds')
axes[1].set_xlabel('')

matches['Year'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='seagreen')
axes[2].set_title('Partidas por Ano')
axes[2].set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
print('Top 15 torneios:')
print(matches['Tournament'].value_counts().head(15).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

matches['Pl 1 hand'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Mão dominante — Player 1 (vencedor)')
axes[0].set_xlabel('')

matches['Best of'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='mediumpurple')
axes[1].set_title('Best of')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

## 2. Overview

In [ ]:
overview.head(3)

In [ ]:
print('Valores únicos em set:', overview['set'].unique())
print('\nNulos:')
print(overview.isnull().sum())

In [ ]:
# Filtrar apenas totais do jogo (não por set)
ov_total = overview[overview['set'] == 'Total'].copy()

# Calcular métricas derivadas
ov_total['first_serve_pct']   = ov_total['first_in']  / ov_total['serve_pts']
ov_total['first_serve_won_pct'] = ov_total['first_won'] / ov_total['first_in'].replace(0, np.nan)
ov_total['second_serve_won_pct'] = ov_total['second_won'] / ov_total['second_in'].replace(0, np.nan)
ov_total['return_won_pct']    = ov_total['return_pts_won'] / ov_total['return_pts'].replace(0, np.nan)
ov_total['bp_saved_pct']      = ov_total['bp_saved'] / ov_total['bk_pts'].replace(0, np.nan)
ov_total['ue_per_pt']         = ov_total['unforced'] / ov_total['serve_pts'].replace(0, np.nan)
ov_total['winners_per_pt']    = ov_total['winners'] / ov_total['serve_pts'].replace(0, np.nan)

print(f'Jogos com dados de Total: {len(ov_total)}')
ov_total[['first_serve_pct','first_serve_won_pct','second_serve_won_pct','return_won_pct','bp_saved_pct']].describe()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

metrics = [
    ('first_serve_pct',      '% 1º Serviço Dentro',     'steelblue'),
    ('first_serve_won_pct',  '% Pontos Ganhos 1º Serve', 'seagreen'),
    ('second_serve_won_pct', '% Pontos Ganhos 2º Serve', 'salmon'),
    ('return_won_pct',       '% Return Points Won',      'mediumpurple'),
    ('bp_saved_pct',         '% Break Points Salvos',    'darkorange'),
    ('winners_per_pt',       'Winners por Ponto',        'crimson'),
]

for ax, (col, title, color) in zip(axes.flatten(), metrics):
    data = ov_total[col].dropna()
    ax.hist(data, bins=40, color=color, edgecolor='white')
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1.5, label=f'Mediana: {data.median():.2f}')
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.suptitle('Distribuição das Métricas — Overview (Total)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ['first_serve_pct','first_serve_won_pct','second_serve_won_pct',
             'return_won_pct','bp_saved_pct','ue_per_pt','winners_per_pt']

corr = ov_total[corr_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True)
plt.title('Correlação entre métricas do Overview')
plt.tight_layout()
plt.show()

## 3. Rally

In [ ]:
rally.head(6)

In [ ]:
print('Valores únicos em row:', rally['row'].unique())

In [ ]:
# Filtrar apenas comprimentos de rali principais (sem sufixo -1)
rally_lengths = rally[rally['row'].isin(['1-3', '4-6', '7-9', '10'])].copy()

# Win % do Player 1 (server) por comprimento de rali
rally_lengths['pl1_win_pct'] = rally_lengths['pl1_won'] / rally_lengths['pts'].replace(0, np.nan)

rally_agg = rally_lengths.groupby('row')['pl1_win_pct'].describe()[['mean','50%','std']]
rally_agg.columns = ['média','mediana','std']
print(rally_agg)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=True)

order = ['1-3', '4-6', '7-9', '10']
colors = ['steelblue', 'seagreen', 'salmon', 'mediumpurple']

for ax, row_val, color in zip(axes, order, colors):
    data = rally_lengths[rally_lengths['row'] == row_val]['pl1_win_pct'].dropna()
    ax.hist(data, bins=30, color=color, edgecolor='white')
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1.5)
    ax.set_title(f'Rali {row_val} shots\nmediana={data.median():.2f}')
    ax.set_xlabel('Win % Player 1')

plt.suptitle('Win % do Player 1 por comprimento de rali', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Total de pontos disputados por comprimento
pts_by_length = rally_lengths.groupby('row')['pts'].sum().reindex(order)

plt.figure(figsize=(8, 4))
pts_by_length.plot(kind='bar', color=colors, edgecolor='white')
plt.title('Total de pontos por comprimento de rali')
plt.xlabel('')
plt.ylabel('Pontos')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. KeyPointsServe

In [ ]:
kp_serve.head(4)

In [ ]:
print('Valores únicos em row:', kp_serve['row'].unique())

In [ ]:
# Filtrar Break Points no saque
kps_bp = kp_serve[kp_serve['row'] == 'BP'].copy()
kps_bp['bp_save_pct'] = kps_bp['pts_won'] / kps_bp['pts'].replace(0, np.nan)
kps_bp['ace_pct']     = kps_bp['aces']    / kps_bp['pts'].replace(0, np.nan)
kps_bp['first_in_pct']= kps_bp['first_in']/ kps_bp['pts'].replace(0, np.nan)

print(kps_bp[['bp_save_pct','ace_pct','first_in_pct']].describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

pairs = [
    ('bp_save_pct',  '% Break Points Salvos no Saque', 'steelblue'),
    ('first_in_pct', '% 1º Serviço Dentro em BP',      'seagreen'),
    ('ace_pct',      '% Aces em BP',                   'salmon'),
]

for ax, (col, title, color) in zip(axes, pairs):
    data = kps_bp[col].dropna()
    ax.hist(data, bins=30, color=color, edgecolor='white')
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1.5, label=f'Mediana: {data.median():.2f}')
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.suptitle('KeyPointsServe — Break Points no Saque', fontsize=13)
plt.tight_layout()
plt.show()

## 5. KeyPointsReturn

In [ ]:
kp_return.head(4)

In [ ]:
print('Valores únicos em row:', kp_return['row'].unique())

In [ ]:
# Filtrar Break Point Opportunities (BPO) — retorno
kpr_bpo = kp_return[kp_return['row'] == 'BPO'].copy()
kpr_bpo['bp_conv_pct']    = kpr_bpo['pts_won']       / kpr_bpo['pts'].replace(0, np.nan)
kpr_bpo['winner_pct']     = kpr_bpo['rally_winners']  / kpr_bpo['pts'].replace(0, np.nan)
kpr_bpo['unforced_pct']   = kpr_bpo['unforced']       / kpr_bpo['pts'].replace(0, np.nan)

print(kpr_bpo[['bp_conv_pct','winner_pct','unforced_pct']].describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

pairs = [
    ('bp_conv_pct',  '% Break Points Convertidos',        'mediumpurple'),
    ('winner_pct',   '% Winners em BPO',                  'darkorange'),
    ('unforced_pct', '% Unforced Errors em BPO (ruim)',   'crimson'),
]

for ax, (col, title, color) in zip(axes, pairs):
    data = kpr_bpo[col].dropna()
    ax.hist(data, bins=30, color=color, edgecolor='white')
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1.5, label=f'Mediana: {data.median():.2f}')
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.suptitle('KeyPointsReturn — Break Point Opportunities', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Visão Consolidada — matches comuns entre datasets

In [ ]:
ids_matches   = set(matches['match_id'])
ids_overview  = set(overview['match_id'])
ids_rally     = set(rally['match_id'])
ids_kp_serve  = set(kp_serve['match_id'])
ids_kp_return = set(kp_return['match_id'])

ids_all = ids_matches & ids_overview & ids_rally & ids_kp_serve & ids_kp_return

print(f'matches:          {len(ids_matches):>6}')
print(f'overview:         {len(ids_overview):>6}')
print(f'rally:            {len(ids_rally):>6}')
print(f'kp_serve:         {len(ids_kp_serve):>6}')
print(f'kp_return:        {len(ids_kp_return):>6}')
print(f'\nComuns (todos 5): {len(ids_all):>6}')

In [ ]:
# Cobertura dos stats por superfície
matches_with_overview = matches[matches['match_id'].isin(ids_overview)]

coverage = pd.DataFrame({
    'Total':    matches['Surface'].value_counts(),
    'Com Overview': matches_with_overview['Surface'].value_counts()
}).fillna(0).astype(int)

coverage['Cobertura %'] = (coverage['Com Overview'] / coverage['Total'] * 100).round(1)
print(coverage)